# Fase 3 — memória de sessão e persistência por cliente

O notebook importa a implementação `.py`, sem duplicar o serviço. A demonstração padrão usa **respostas e embeddings roteirizados**, identificados; não comprova compreensão de linguagem. A execução real separada fica em `04_avaliacao_e_evidencias/resultados`.

Uma sessão tem uma janela de turnos completos. O perfil confirmado e resumos autorizados persistem por cliente em SQLite. Abrir outra sessão não apaga esses dados; `/esquecer` apaga os dois tipos de memória e o histórico daquele cliente. A identidade é de demonstração, sem autenticação.

In [1]:
# AMBIENTE: encontrar o pacote por marcador torna a cópia do projeto portátil.
from pathlib import Path
import os
import sys

inicio = Path.cwd().resolve()
projeto = next(p for p in (inicio, *inicio.parents) if (p / 'INSTRUCOES_PARA_IA_PROGRAMADORA.md').exists())
sys.path.insert(0, str(projeto / '02_codigo/src'))
print('Python:', sys.executable)
print('Projeto:', projeto.name)
modo = os.environ.get('FASE03_MODO', 'simulado')
print('Modo explicitamente selecionado:', modo)


Python: C:\Users\Renan\Documents\GIT_PROJETOS\estudos-engenharia-ia\.venv\Scripts\python.exe
Projeto: Projeto final
Modo explicitamente selecionado: simulado


## Fluxo realmente implementado

1. O controlador escolhe `cliente_id` e `session_id` e confere a propriedade da sessão.
2. Lê os últimos seis turnos completos, o perfil confirmado e resumos relevantes filtrados por cliente/modelo.
3. `invoke` recebe o contexto e a pergunta atual uma vez. Ferramentas continuam sendo a fonte técnica/comercial.
4. Só registra o par pergunta/resposta após sucesso. Perfil/resumo não é extraído automaticamente da resposta da LLM.

A busca vetorial usa cosseno em SQLite para a pequena coleção de resumos; não é FAISS nem RAG de políticas. `text-embedding-3-small` é usado no modo real; no modo simulado os vetores são fixtures.

In [2]:
# INVOKE/STORE: o mesmo módulo executa os casos no terminal e no notebook.
from auto_super_parts.demonstracao_memoria import demonstrar

resultado = demonstrar(modo)
print('Compreensão por LLM real avaliada nesta execução:', not resultado['modelos_simulados'])


informar_na_sessao OK


lembrar_na_sessao OK


nova_sessao_perfil_e_resumo OK


outro_cliente_sem_vazamento OK


correcao_persistida OK


apos_exclusao OK


Compreensão por LLM real avaliada nesta execução: False


In [3]:
# EVIDÊNCIAS: conferimos observado por caso, sem transformar fixture em resultado real.
for caso in resultado['casos']:
    r = caso['resultado']
    print()
    print('Caso:', caso['caso'])
    print('Critério local atendido:', caso['criterio_local_atendido'])
    print('Resposta:', r.get('resposta'))
    print('Mensagens do histórico enviadas:', len(r['historico_enviado']))
    print('Recuperação:', r['memoria_utilizada']['recuperacao']['status'])



Caso: informar_na_sessao
Critério local atendido: True
Resposta: [SIMULAÇÃO] Anotei o Fusca nesta conversa.
Mensagens do histórico enviadas: 0
Recuperação: sem_resumos

Caso: lembrar_na_sessao
Critério local atendido: True
Resposta: [SIMULAÇÃO] Você informou Fusca 1982 1600 a ar gasolina.
Mensagens do histórico enviadas: 2
Recuperação: sem_resumos

Caso: nova_sessao_perfil_e_resumo
Critério local atendido: True
Resposta: [SIMULAÇÃO] O veículo salvo é Celta 2012; uso urbano em percursos curtos.
Mensagens do histórico enviadas: 0
Recuperação: ok

Caso: outro_cliente_sem_vazamento
Critério local atendido: True
Resposta: [SIMULAÇÃO] Você ainda não informou veículo neste cliente.
Mensagens do histórico enviadas: 0
Recuperação: sem_resumos

Caso: correcao_persistida
Critério local atendido: True
Resposta: [SIMULAÇÃO] O veículo salvo agora é S10 2014 diesel.
Mensagens do histórico enviadas: 0
Recuperação: ok

Caso: apos_exclusao
Critério local atendido: True
Resposta: [SIMULAÇÃO] Não tenho v

In [4]:
# PROCESSO NOVO: ler o arquivo em outro interpretador verifica persistência em disco.
assert resultado['persistencia_ok']
assert resultado['recuperacao_vetorial_executada']
assert all(c['criterio_local_atendido'] for c in resultado['casos'])
print('Perfil A no processo separado:', resultado['persistencia_processo_separado']['A']['veiculo_confirmado']['modelo'])
print('Perfil B:', resultado['persistencia_processo_separado']['B'])
print('Esta execução foi simulada:', resultado['modelos_simulados'])


Perfil A no processo separado: Celta
Perfil B: None
Esta execução foi simulada: True


## Experimentar de verdade

Na pasta `Projeto final`, execute:

```powershell
python -B 02_codigo/src/auto_super_parts/atendimento_memoria.py --cliente cliente_demo
```

Diga o veículo e pergunte por ele; use `/veiculos`, confirme com `/veiculo V003`, abra `/nova` e pergunte de novo. Para resumos: `/guardar Uso urbano com percursos curtos`. Troque com `/cliente outro_cliente` e confira isolamento; volte ao cliente original em nova sessão. `/esquecer` remove perfil, resumos e históricos daquele cliente.

Para repetir os casos com chat e embeddings reais: `python -B 02_codigo/validar_memoria.py --real`. É necessário configurar a chave local do projeto e há consumo de API. O banco temporário da demonstração é descartado; o banco da CLI fica em `02_codigo/estado_local`, fora do Git e do ZIP.

## Limites

Lembrar na mesma sessão não comprova recuperação após reiniciar: pode ser só histórico em RAM. A prova separada fecha o SQLite e lê o perfil em outro processo antes de abrir nova sessão. Não é autenticação, recuperação de toda a conversa antiga, memória infinita ou melhoria automática de regras. Não há resumo automático por LLM nem aprendizagem procedural; o prompt/código definem os procedimentos. FAISS/políticas e verificador semântico geral pertencem aos próximos incrementos.